# D3_02 불균형 대응과 과대적합 진단

> **오늘 질문:** 고장을 더 많이 잡으려면 무엇을 바꿔야 할까?

1교시의 의사결정나무는 평가 기간 고장 39건 중 8건만 잡았습니다(재현율 0.21). 세 가지 방법으로 이 숫자를 올려 봅니다.

| 방법 | 하는 일 |
|---|---|
| 클래스 가중치 | 고장을 틀리면 더 큰 벌점 |
| Under / Over Sampling | 정상을 줄이거나, 고장을 복제 |
| SMOTE | 고장과 비슷한 새 데이터를 만듦 |

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비: 1교시와 같은 데이터·분할 📋

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from imblearn.pipeline import make_pipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE

df = pd.read_parquet("../../data/factory_table.parquet")
features = ["air_temp", "process_temp", "rpm", "torque", "tool_wear"]
X = df[features]
y = df["failure"]

cut = int(len(df) * 0.8)                       # 앞 80% 학습 / 마지막 20% 평가
X_train, X_test = X.iloc[:cut], X.iloc[cut:]
y_train, y_test = y.iloc[:cut], y.iloc[cut:]

## 1. 클래스 가중치 ✍️

`class_weight="balanced"`는 드문 클래스(고장)를 틀릴 때 더 큰 벌점을 줍니다. 옵션 하나로 끝나는 가장 쉬운 첫 시도입니다.

> 변수 이름: `tree_w`

In [ ]:
# 힌트: DecisionTreeClassifier(..., class_weight="balanced")

## 2. 리샘플링 결과 눈으로 보기 ✍️

SMOTE를 Train에 적용하면 고장 건수가 정상과 같아집니다. **Test에는 절대 적용하지 않습니다.**

> 변수 이름: `X_res`, `y_res`

In [ ]:
# 힌트: SMOTE(random_state=42).fit_resample(X, y)

## 3. 다섯 가지 방법 비교 📋

`make_pipeline(리샘플러, 모델)`로 묶으면 리샘플링이 학습 데이터에만 적용됩니다. 같은 Test로 비교합니다.

In [ ]:
methods = {
    "원본": DecisionTreeClassifier(max_depth=5, random_state=42),
    "클래스 가중치": DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced"),
    "Under": make_pipeline(RandomUnderSampler(random_state=42), DecisionTreeClassifier(max_depth=5, random_state=42)),
    "Over": make_pipeline(RandomOverSampler(random_state=42), DecisionTreeClassifier(max_depth=5, random_state=42)),
    "SMOTE": make_pipeline(SMOTE(random_state=42), DecisionTreeClassifier(max_depth=5, random_state=42)),
}

rows = []
for name, model in methods.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({"방법": name,
                 "정밀도": precision_score(y_test, pred),
                 "재현율": recall_score(y_test, pred),
                 "F1": f1_score(y_test, pred)})
pd.DataFrame(rows).round(2)

**Q1.** 원본과 비교해 재현율과 정밀도는 어떻게 바뀌었나요? 고장을 놓치면 라인이 멈추는 현장이라면 어떤 방법을 고르겠습니까?

> 답:


## 4. 과대적합 진단 ✍️

깊이 제한 없이 나무를 끝까지 키우고, **Train 점수와 Test 점수**를 나란히 봅니다.

> 변수 이름: `deep`

In [ ]:
# 힌트: f1_score(정답, 예측)를 Train과 Test에 각각

비교를 위해 깊이 5 나무의 Train·Test 점수도 봅니다.

## 정리 💬

**Q2.** 끝까지 키운 나무와 깊이 5 나무를 Train·Test 점수로 진단해 보세요. 어느 쪽이 과대적합이고, 어느 쪽이 과소적합에 가까운가요?

> 답:
